# MAKE SURE TO CHECK AND CHANGE THIS CELL BEFORE EVERY RUN

## CURRENTLY RUNNING: WHISPER LARGE V3 TURBO

In [1]:
%%capture
!pip install transformers datasets peft accelerate bitsandbytes evaluate jiwer trl pydub

In [2]:
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()
secret_value_0 = user_secrets.get_secret("HF_TOKEN")
secret_value_1 = user_secrets.get_secret("HF_WRITE")

In [3]:
from huggingface_hub import login

login(secret_value_0)

In [4]:
#final_data check
from datasets.load import load_dataset

dataset = load_dataset("KaliNangia/Punjabi-STT_Combined_Preprocessed")

README.md:   0%|          | 0.00/469 [00:00<?, ?B/s]

test-00000-of-00006.parquet:   0%|          | 0.00/262M [00:00<?, ?B/s]

test-00001-of-00006.parquet:   0%|          | 0.00/259M [00:00<?, ?B/s]

test-00002-of-00006.parquet:   0%|          | 0.00/246M [00:00<?, ?B/s]

test-00003-of-00006.parquet:   0%|          | 0.00/253M [00:00<?, ?B/s]

...

train-00010-of-00013.parquet:   0%|          | 0.00/257M [00:00<?, ?B/s]

train-00011-of-00013.parquet:   0%|          | 0.00/261M [00:00<?, ?B/s]

train-00012-of-00013.parquet:   0%|          | 0.00/272M [00:00<?, ?B/s]

Generating test split:   0%|          | 0/1791 [00:00<?, ? examples/s]

Generating train split:   0%|          | 0/3931 [00:00<?, ? examples/s]

In [5]:
from transformers import WhisperFeatureExtractor, WhisperTokenizer, WhisperProcessor

#Set the model
model_id = "openai/whisper-large-v3"

# Load preprocessors
feature_extractor = WhisperFeatureExtractor.from_pretrained(model_id)
# Set language to punjabi and task to transcribe
tokenizer = WhisperTokenizer.from_pretrained(model_id, language="punjabi", task="transcribe")
processor = WhisperProcessor.from_pretrained(model_id, language="punjabi", task="transcribe")

max_label_length = 448

preprocessor_config.json:   0%|          | 0.00/340 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

normalizer.json: 0.00B [00:00, ?B/s]

added_tokens.json: 0.00B [00:00, ?B/s]

special_tokens_map.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

In [6]:
dataset

DatasetDict({
    test: Dataset({
        features: ['input_features', 'labels'],
        num_rows: 1791
    })
    train: Dataset({
        features: ['input_features', 'labels'],
        num_rows: 3931
    })
})

In [7]:
import torch
from dataclasses import dataclass
from typing import Any, Dict, List, Union

@dataclass
class DataCollatorSpeechSeq2SeqWithPadding:
    processor: Any
    decoder_start_token_id: int = 50258  # Default start token for Whisper models
    pad_token_id: int = -100

    def __call__(self, features: List[Dict[str, Union[List[int], torch.Tensor]]]) -> Dict[str, torch.Tensor]:
        # 1. Pad audio input features
        input_features = [{"input_features": feature["input_features"]} for feature in features]
        batch = self.processor.feature_extractor.pad(input_features, return_tensors="pt")

        # 2. Pad text label features
        label_features = [{"input_ids": feature["labels"]} for feature in features]
        labels_batch = self.processor.tokenizer.pad(label_features, return_tensors="pt")

        # Mask padding with -100
        labels = labels_batch["input_ids"].masked_fill(labels_batch.attention_mask.ne(1), -100)

        # Truncate BOS token if present
        if (labels[:, 0] == self.processor.tokenizer.bos_token_id).all().cpu().item():
            labels = labels[:, 1:]

        batch["labels"] = labels

        # --- THE FIX: MANUALLY COMPUTE DECODER INPUT IDS ---
        # Shift the labels to the right to construct the perfect initial decoder state
        shifted_labels = labels.new_zeros(labels.shape)
        shifted_labels[:, 1:] = labels[:, :-1].clone()
        shifted_labels[:, 0] = self.decoder_start_token_id
        
        # Replace any remaining -100 values in decoder inputs with the processor's true pad ID
        true_pad_token_id = self.processor.tokenizer.pad_token_id
        shifted_labels.masked_fill_(shifted_labels == -100, true_pad_token_id)
        
        batch["decoder_input_ids"] = shifted_labels

        return batch

data_collator = DataCollatorSpeechSeq2SeqWithPadding(processor=processor)

In [8]:
from transformers import WhisperForConditionalGeneration, BitsAndBytesConfig
from peft import LoraConfig, get_peft_model, prepare_model_for_kbit_training
import torch


model_id = "openai/whisper-large-v3"

# 1. Define the 8-bit quantization configuration
quantization_config = BitsAndBytesConfig(
    load_in_8bit=True
)

# 2. Pass the config to the model initialization
model = WhisperForConditionalGeneration.from_pretrained(
    model_id,
    quantization_config=quantization_config,  # Use this instead of load_in_8bit=True
    device_map="auto"  # Remove so that model utilizes both available GPUS
)

# Prepare model for PEFT int8 training quirks
model = prepare_model_for_kbit_training(model)

# Define LoRA Configuration
config = LoraConfig(
    r=16,                                # Dropped from 32 to 16 to reduce capacity (less overfitting)
    lora_alpha=32,                       # Scaled proportionally (r * 2)
    target_modules=["q_proj", "v_proj", "k_proj", "out_proj", "fc1", "fc2"],
    lora_dropout=0.1,                    # Increased from 0.05 to 0.1 to drop out more activations
    bias="none"
)

model = get_peft_model(model, config)
model.print_trainable_parameters()
# You will notice that < 2% of the parameters are actually trainable now!

model.safetensors:   0%|          | 0.00/3.09G [00:00<?, ?B/s]

Loading weights:   0%|          | 0/1259 [00:00<?, ?it/s]

generation_config.json: 0.00B [00:00, ?B/s]

trainable params: 28,835,840 || all params: 1,572,326,400 || trainable%: 1.8340


In [9]:
# # FOR COLAB
# from google.colab import userdata
# HF_WRITE = userdata.get('HF_DATASET_TOKEN')

In [11]:
from transformers import Seq2SeqTrainingArguments, EarlyStoppingCallback, Seq2SeqTrainer

training_args = Seq2SeqTrainingArguments(
    output_dir="./whisper-large-v3-gurmukhi-lora",
    per_device_train_batch_size=4,
    gradient_accumulation_steps=8,
    learning_rate=5e-5,                  # 1. LOWER LEARNING RATE to slow down overfitting
    warmup_steps=100,
    max_steps=1000,

    # --- STRICT EVAL & SAVE ALIGNMENT FOR EARLY STOPPING ---
    eval_strategy="steps",
    eval_steps=50,                       # Check validation loss every 50 steps
    save_strategy="steps",
    save_steps=50,                       # MUST match eval_steps
    save_total_limit=2,                  # Only keeps the best and most recent checkpoints to save space

    # --- REGULARIZATION ---
    weight_decay=0.05,                   # Adds L2 regularization penalty to weights
    label_smoothing_factor=0.1,          # Prevents the model from over-allocating confidence to tokens

    predict_with_generate=False,
    logging_steps=10,

    fp16=True,
    gradient_checkpointing=True,
    optim="adamw_bnb_8bit",

    # --- HUB SETTINGS ---
    push_to_hub=True,                    # Enable uploading to HF Hub
    hub_model_id="abhi8799/whisper-large-v3-gurmukhi-lora", # Target repo ID
    hub_strategy="every_save",           # Upload checkpoints as they are saved
    hub_token= secret_value_1,

    # --- CRITICAL ---
    load_best_model_at_end=True,         # Required for Early Stopping to pull the absolute lowest loss checkpoint
    metric_for_best_model="loss",        # Track loss
    greater_is_better=False              # Lower loss is better
)

# Fix configurations for generating text during evaluation
model.config.forced_decoder_ids = None
model.config.suppress_tokens = []

trainer = Seq2SeqTrainer(
    args=training_args,
    model=model,
    train_dataset=dataset["train"],
    eval_dataset=dataset["test"],
    data_collator=data_collator,
    processing_class=processor,
    callbacks=[EarlyStoppingCallback(early_stopping_patience=3)]
)

In [ ]:
# 1. Start training
trainer.train()

# 2. Push the final trained LoRA adapters and tokenizer configurations to the Hub
trainer.push_to_hub(tags="automatic-speech-recognition", commit_message = "Finally Commited. End of training")

The tokenizer has new PAD/BOS/EOS tokens that differ from the model config and generation config. The model config and generation config were aligned accordingly, being updated with the tokenizer's values. Updated tokens: {'pad_token_id': 50257}.
/usr/local/lib/python3.12/dist-packages/torch/utils/data/dataloader.py:775: UserWarning: 'pin_memory' argument is set as true but no accelerator is found, then device pinned memory won't be used.
  super().__init__(loader)


If the model hasn't completely trained and stops midway, store and push current model which has been safely stored on HF hub

In [ ]:
# !unzip results.zip -d /kaggle/working/

In [ ]:
# from transformers import WhisperForConditionalGeneration, BitsAndBytesConfig
# from peft import PeftModel
# from huggingface_hub import login

# model_id = "openai/whisper-small" ###############################################################################

# # 1. Log in to your Hugging Face account
# # (Make sure to use a token with WRITE permissions)
# login(secret_value_1)#########################################################################################

# # 2. Re-define the quantization config just in case the notebook state cleared it
# quantization_config = BitsAndBytesConfig(load_in_8bit=True)

# # 3. Load the base model
# base_model = WhisperForConditionalGeneration.from_pretrained(
#     model_id,
#     quantization_config=quantization_config,
#     device_map="auto"
# )

# # 4. Point to your highest checkpoint number in Kaggle's directory
# # Change 'checkpoint-1000' to whatever your highest number is
# final_checkpoint_path = "/kaggle/working/whisper-small-gurmukhi-lora/checkpoint-1000" ####################################################

# # 5. Combine the base model with your trained LoRA weights
# model = PeftModel.from_pretrained(base_model, final_checkpoint_path)

# # 6. Push everything to the Hub securely
# repo_id = "abhi8799/whisper-small-gurmukhi-lora" ################################################################

# print("Uploading weights...")
# model.push_to_hub(repo_id, commit_message="End of fine-tuning on Gurmukhi dataset")

# print("Uploading processor...")
# processor.push_to_hub(repo_id)

# print("Done! Check your Hugging Face profile.")